# هل تساعد معاينة الكتاب القارئ على اختيار ما يحفظه؟

## 0. بيانات المشروع

| البند | القيمة |
|---|---|
| المشروع | معاينة المحتوى في نتائج البحث عن الكتب |
| اسم الفريق | A&Y&F |
| أعضاء الفريق | 1. عبدالوهاب النصار، 2. يوسف المطيري، 3. فراس الناصر |
| المنتج | مكتبة كتب إلكترونية افتراضية تتيح البحث وحفظ الكتب في قائمة القراءة |
| المرحلة | اليوم الأول من مشروع التجارب واختبارات A/B والاستدلال السببي |

## 1. السؤال والفرضية

**القرار:** هل نعرض مقتطفًا قصيرًا من وصف الكتاب داخل بطاقة نتيجة البحث؟
**A:** تعرض البطاقة الغلاف والعنوان والمؤلف. **B:** تعرض العناصر نفسها مع مقتطف من الوصف لا يتجاوز سطرين. ترتيب الكتب والأزرار ثابت في النسختين.

**الفرضية:** إذا ظهر المقتطف في نتائج البحث، فستزيد نسبة المستخدمين الذين يحفظون كتابًا خلال 24 ساعة من أول بحث مؤهل، لأن المقتطف يساعدهم على تقدير ملاءمة الكتاب قبل فتح صفحته.

**ما الذي يغيّر رأينا؟** نقترح الإطلاق إذا ارتفع معدل الحفظ بمقدار نقطتين مئويتين على الأقل، وكان الحد الأدنى لفترة الثقة 95% لفرق B عن A أعلى من الصفر، وبقيت مقاييس الحماية ضمن الحدود أدناه. إذا لم تتحقق الشروط، نؤجل الإطلاق. هذه حدود قرار مقترحة قبل جمع البيانات.

## 2. تصميم التجربة

| البند | التعريف |
|---|---|
| الأهلية وبداية القياس | نُدخل المستخدم المسجّل عند أول بحث يعرض كتابًا واحدًا على الأقل له وصف، ونحدد أهليته قبل عرض النسخة. |
| وحدة العشوأة | حساب المستخدم. يبقى الحساب في النسخة نفسها عبر الزيارات والأجهزة. |
| نسبة التوزيع | 50% لكل نسخة باستخدام تجزئة ثابتة لمعرّف التجربة والحساب. |
| المقياس الرئيسي | عدد المستخدمين الموزّعين الذين حفظوا كتابًا من نتائج البحث خلال 24 ساعة ÷ جميع المستخدمين الموزّعين الذين اكتملت نافذتهم. يُحسب المستخدم مرة واحدة مهما تعددت عمليات الحفظ. |
| مقاييس ثانوية | نسبة المستخدمين الذين فتحوا صفحة كتاب من النتائج خلال 24 ساعة، ومتوسط عدد الكتب المختلفة المحفوظة لكل مستخدم موزّع خلال النافذة نفسها. |
| مقاييس الحماية | زمن عرض النتائج عند المئين 95: نسمح بزيادة حتى 150 مللي ثانية. معدل طلبات البحث الفاشلة: نسمح بزيادة حتى 0.2 نقطة مئوية. نقارن B مع A. |
| شرط SUTVA | قائمة القراءة خاصة بكل حساب. نثبّت ترتيب النتائج ونعطّل إعادة الترتيب المبنية على نشاط المشاركين حتى لا يغيّر حفظ مستخدم نتائج مستخدم آخر. |
| فحص الثقة قبل التحليل | نتحقق من ثبات التخصيص، ثم نفحص SRM على أعداد الحسابات الموزّعة مقابل 50/50. إذا كانت قيمة p أقل من 0.001 نراجع التوزيع والتسجيل قبل تفسير الأثر. |

خلية التحقق أدناه تستخدم معرّفات اصطناعية لفحص آلية التخصيص فقط. يبدأ المشروع هنا بتصميم التجربة، وتُحدَّد العينة والمدة في اليوم الثاني.

In [1]:
from hashlib import sha256

def assign_variant(user_id):
    key = f"book-search-preview-v1:{user_id}".encode("utf-8")
    bucket = int.from_bytes(sha256(key).digest()[:8], "big")
    return "B" if bucket < 2**63 else "A"

# معرّفات اختبار مستقلة عن بيانات القالب.
test_ids = [f"demo-{i}" for i in range(20)]
first_visit = {uid: assign_variant(uid) for uid in test_ids}
second_visit = {uid: assign_variant(uid) for uid in reversed(test_ids)}
assert first_visit == second_visit
assert set(first_visit.values()) == {"A", "B"}
print("نجح اختبار التخصيص: بقيت الحسابات العشرون في نسخها عند إعادة الزيارة.")

نجح اختبار التخصيص: بقيت الحسابات العشرون في نسخها عند إعادة الزيارة.


## 3. اليوم الثاني | حجم العينة ومدة تجربة معاينة الكتب

نحسب عدد **الحسابات** اللازمة لمقارنة نسبة من يحفظون كتابًا خلال 24 ساعة من أول بحث مؤهل. يظهر المقتطف للنسخة B، وتبقى النسخة A على التصميم الحالي.

| مدخل التخطيط | القيمة | أساس الاختيار |
|---|---:|---|
| معدل الحفظ في A | 18% | افتراض تخطيطي للمنتج الافتراضي؛ يُراجع من سجلّات ما قبل التجربة عند توفرها. |
| أصغر زيادة مطلوبة | نقطتان مئويتان، من 18% إلى 20% | حد الأثر العملي الذي حدده الفريق في اليوم الأول؛ يعادل زيادة نسبية 11.1%. |
| تدفّق المشاركين | 800 حساب مؤهل جديد يوميًا إجمالًا | افتراض تخطيطي؛ يُحسب الحساب مرة واحدة عند أول بحث مؤهل. |
| التوزيع | 50% لكل نسخة | نفس التخصيص الثابت على مستوى الحساب في اليوم الأول. |
| الدلالة والقوة | 5% لاختبار ثنائي الطرف، وقوة 80% | تُحسب العينة لاكتشاف فرق عن الصفر عندما يكون الأثر الحقيقي نقطتين. تبقى شروط الإطلاق ومقاييس الحماية كما حُدّدت في اليوم الأول. |

**قرار الفريق.** نعتمد **21 يومًا للتسجيل** تغطي ثلاث دورات أسبوعية كاملة، ثم ننتظر **24 ساعة** حتى تكتمل نافذة الحفظ لآخر مشارك. الحد الأدنى **6,036 حسابًا لكل نسخة**، ونتوقع **8,400** خلال الخطة. أسبوع واحد يوفّر نحو 2,800 حساب لكل نسخة، ويكشف فرقًا يقارب **2.88 نقطة**، أكبر من هدفنا. خلال المدة المخططة ينخفض هذا الرقم إلى **1.66 نقطة**. المدة مقبولة في هذا السيناريو؛ وإذا انخفض التدفق ولم نصل إلى الحد الأدنى في كل مجموعة، نمدّد التسجيل بأسابيع كاملة وفق أعداد المشاركين، دون استخدام نتائج الحفظ لاتخاذ قرار التمديد.

الحساب التالي يستخدم تقريب النسبتين ودالة أصغر أثر قابل للاكتشاف الواردين في التدريب. القيم أعلاه تخص التخطيط، ولا تمثّل نتائج تجربة منفّذة.

In [2]:
from math import ceil, sqrt
from statistics import NormalDist

CONFIG = {
    "baseline_save_rate": 0.18,
    "mde_abs": 0.02,
    "daily_eligible_accounts": 800,
    "treatment_share": 0.50,
    "alpha": 0.05,
    "power": 0.80,
    "follow_up_hours": 24,
}

p_a = CONFIG["baseline_save_rate"]
p_b = p_a + CONFIG["mde_abs"]
daily = CONFIG["daily_eligible_accounts"]
assert CONFIG["treatment_share"] == 0.5  # الصيغة لمجموعتين متساويتين
normal = NormalDist()
z_sum = normal.inv_cdf(1 - CONFIG["alpha"] / 2) + normal.inv_cdf(CONFIG["power"])
variance = p_a * (1 - p_a) + p_b * (1 - p_b)
n_min = ceil(z_sum**2 * variance / CONFIG["mde_abs"]**2)
days_min = ceil(2 * n_min / daily)
weeks = ceil(days_min / 7)
days = 7 * weeks

print(f"الحد الأدنى: {n_min:,} حسابًا لكل نسخة، أي {2 * n_min:,} إجمالًا.")
print(f"التسجيل: {days_min} يومًا قبل التقريب، و{days} يومًا ({weeks} أسابيع) بعده.")
for label, period in [("مدة الخطة", days), ("أسبوع واحد", 7)]:
    per_group = period * daily * CONFIG["treatment_share"]
    mde = z_sum * sqrt(2 * p_a * (1 - p_a) / per_group)
    print(f"{label}: {per_group:,.0f} حساب لكل نسخة، وأصغر أثر تقريبي {100 * mde:.2f} نقطة مئوية.")
print(f"موعد التحليل: بعد {days} يوم تسجيل و{CONFIG['follow_up_hours']} ساعة متابعة لآخر حساب.")

الحد الأدنى: 6,036 حسابًا لكل نسخة، أي 12,072 إجمالًا.
التسجيل: 16 يومًا قبل التقريب، و21 يومًا (3 أسابيع) بعده.
مدة الخطة: 8,400 حساب لكل نسخة، وأصغر أثر تقريبي 1.66 نقطة مئوية.
أسبوع واحد: 2,800 حساب لكل نسخة، وأصغر أثر تقريبي 2.88 نقطة مئوية.
موعد التحليل: بعد 21 يوم تسجيل و24 ساعة متابعة لآخر حساب.
